# Паттерн 5: Pipeline — линейный конвейер

Pipeline — самый простой паттерн оркестрации в мультиагентных системах. Агенты выстроены в цепочку: каждый обрабатывает результат предыдущего и передаёт следующему. Нет условий, нет циклов — чистая последовательность. Каждый агент заполняет своё поле в общем состоянии: никаких reducer-ов, никаких конфликтов — каждое поле пишет ровно один агент.

В этом примере четыре агента-специалиста последовательно готовят статью: исследователь собирает факты, аналитик выделяет тренды, писатель создаёт связный текст, редактор полирует результат.

```mermaid
graph LR
    START((START)) --> researcher(["🔹 Researcher<br/><i>ищет</i>"])
    researcher --> analyst(["🔹 Analyst<br/><i>анализирует</i>"])
    analyst --> writer(["🔹 Writer<br/><i>пишет</i>"])
    writer --> editor(["🔹 Editor<br/><i>редактирует</i>"])
    editor --> END((END))

    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class researcher,analyst,writer,editor worker
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние конвейера

Каждый агент в Pipeline заполняет ровно одно поле в общем состоянии. Исходная тема приходит в `topic`, а далее каждый этап записывает свой результат: `research`, `analysis`, `draft`, `final_text`. Никаких аддитивных reducer-ов (как `operator.add` в Supervisor) здесь не нужно — конфликтов записи нет по определению, потому что каждое поле пишет строго один агент.

In [3]:
llm = get_llm()

class ArticleState(TypedDict):
    topic: str          # Исходная тема
    research: str       # Результат исследователя
    analysis: str       # Результат аналитика
    draft: str          # Черновик писателя
    final_text: str     # Финальный текст редактора

## Этап 1: Исследователь

Первый узел конвейера — исследователь. Он получает тему из состояния и собирает ключевые факты. Результат записывается в поле `research`, которое следующий агент (аналитик) прочитает как входные данные.

In [4]:
def researcher(state: ArticleState) -> dict:
    """Этап 1: Исследование — сбор фактов по теме."""
    response = llm.invoke([
        SystemMessage(content="Ты — исследователь. Найди 3-4 ключевых факта по теме. Кратко, по пунктам."),
        HumanMessage(content=state["topic"])
    ])
    print(f"  [1/4 Исследователь] Найдены факты ({len(response.content)} символов)")
    return {"research": response.content}

## Этап 2: Аналитик

Аналитик получает и тему, и собранные факты. Его задача — выделить тренды, закономерности и ключевые выводы. Обратите внимание: он читает поле `research`, заполненное на предыдущем этапе, и записывает результат в своё собственное поле `analysis`.

In [5]:
def analyst(state: ArticleState) -> dict:
    """Этап 2: Анализ — выводы и интерпретация фактов."""
    response = llm.invoke([
        SystemMessage(content="Ты — аналитик. Проанализируй факты, выдели тренды и ключевые выводы. Кратко."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nФакты:\n{state['research']}")
    ])
    print(f"  [2/4 Аналитик] Анализ готов ({len(response.content)} символов)")
    return {"analysis": response.content}

## Этап 3: Писатель

Писатель — третье звено конвейера. Он получает исходную тему, собранные факты и аналитические выводы, а на их основе создаёт связный текст-черновик. Результат попадает в поле `draft`.

In [6]:
def writer(state: ArticleState) -> dict:
    """Этап 3: Написание — создание связного текста."""
    response = llm.invoke([
        SystemMessage(content="Ты — писатель. Напиши связный текст (4-5 предложений) на основе исследования и анализа."),
        HumanMessage(content=f"Тема: {state['topic']}\n\nФакты:\n{state['research']}\n\nАнализ:\n{state['analysis']}")
    ])
    print(f"  [3/4 Писатель] Черновик готов ({len(response.content)} символов)")
    return {"draft": response.content}

## Этап 4: Редактор

Финальное звено конвейера — редактор. Он получает черновик писателя и улучшает стиль, исправляет ошибки, делает текст более читаемым. Результат записывается в `final_text` — это итоговый выход всего Pipeline.

In [7]:
def editor(state: ArticleState) -> dict:
    """Этап 4: Редактура — финальная обработка текста."""
    response = llm.invoke([
        SystemMessage(content="Ты — редактор. Улучши стиль, исправь ошибки, сделай текст более читаемым. Верни финальный текст."),
        HumanMessage(content=state["draft"])
    ])
    print(f"  [4/4 Редактор] Финальный текст готов ({len(response.content)} символов)")
    return {"final_text": response.content}

## Сборка графа

Главное отличие Pipeline от всех остальных паттернов — здесь используются только простые рёбра `add_edge`. Никаких `add_conditional_edges`, никакой маршрутизации. Четыре узла, пять рёбер, строго линейная цепочка: START -> researcher -> analyst -> writer -> editor -> END. Это самый предсказуемый и дешёвый паттерн — ровно четыре вызова LLM, без вариаций.

In [8]:
graph = StateGraph(ArticleState)

graph.add_node("researcher", researcher)
graph.add_node("analyst", analyst)
graph.add_node("writer", writer)
graph.add_node("editor", editor)

# Линейная цепочка — простые рёбра, никаких условий
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", "editor")
graph.add_edge("editor", END)

app = graph.compile()

## Запуск

Запускаем конвейер с темой. Каждый агент выполнится строго по порядку: исследователь -> аналитик -> писатель -> редактор. В логах видно, как результат постепенно обогащается на каждом этапе. Никакого `recursion_limit` не требуется — циклов в графе нет.

In [9]:
print("\n" + "=" * 60)
print("ПРИМЕР: Pipeline — конвейерная обработка")
print("=" * 60)

result = app.invoke({
    "topic": "Влияние искусственного интеллекта на рынок труда",
    "research": "", "analysis": "", "draft": "", "final_text": ""
})

print(f"\n  Финальный текст:\n{result['final_text']}")


ПРИМЕР: Pipeline — конвейерная обработка


  [1/4 Исследователь] Найдены факты (1015 символов)


  [2/4 Аналитик] Анализ готов (776 символов)


  [3/4 Писатель] Черновик готов (696 символов)


  [4/4 Редактор] Финальный текст готов (706 символов)

  Финальный текст:
Искусственный интеллект заметно меняет рынок труда, прежде всего за счёт автоматизации рутинных задач в производстве, клиентской поддержке, анализе данных и документообороте. При этом он не столько уничтожает рабочие места, сколько трансформирует их: человеку всё чаще остаются контроль, принятие решений и сложные коммуникации. На фоне этих изменений растёт спрос на новые навыки — работу с данными, цифровыми инструментами, кибербезопасностью, критическое мышление и способность быстро переучиваться. Однако выгоды от внедрения ИИ распределяются неравномерно, поэтому высококвалифицированные специалисты и технологически сильные компании получают больше преимуществ, чем работники с низкой квалификацией.


## Итоги

**Pipeline** — самый простой и предсказуемый паттерн мультиагентной оркестрации:

- **Структура**: линейная цепочка агентов, соединённых простыми рёбрами `add_edge`
- **Состояние**: каждый агент пишет в своё поле, reducer-ы не нужны
- **Стоимость**: фиксированная — ровно N вызовов LLM (по числу агентов в цепочке)
- **Когда применять**: задачи с чётко определённой последовательностью этапов — подготовка контента, ETL-пайплайны, многоступенчатая обработка текста
- **Ограничения**: нет обратной связи между этапами, нет возможности пропустить шаг или вернуться назад